# 9j — Forecast assembly, WIS scoring & diagnostics (two-stage cut)

Downstream half of the preliminary composable forecast (**four ways**), split out from
`8j_preliminary_forecast.ipynb`. **8j fits and caches the two-stage artefacts** — Stage-1 GP
chains `../dt_intermediate/8j_s1_*.jld2` and Stage-2 pooled infection draws `8j_s2_*.jld2`; this
notebook **reloads them** (no re-fit) to assemble the pooled contact-updated forecasts (10 000
draws per origin×combo×horizon), score them with **log-scale WIS** via R `scoringutils`, and draw
the diagnostic figures. **Run 8j first** — a missing artefact here only triggers a fallback re-fit.

The shared setup (`cfg`, `grid`, `raw`, `FORECAST_ORIGINS`, `wins`, `combos`) is reproduced
verbatim from 8j so the cache keys `(degree[, ngm], contacts, origin, h)` match exactly.
Outputs keep the `8j_` prefix (`res/8j_*`).

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework

using Random, Statistics
mkpath("../res")

default_plot_setting()

In [ ]:
# `constant_contacts = false` ⇒ contact degree estimated PER WEEK, temporally smoothed by a
# separable spatio-temporal GP (shared ρ_diag/ρ_gap/ρ_time, η, σ_c; scalar intercept c +
# temporal-level GP cₜ = c + σ_c·(Lt·z_c) + matrix-normal field η·Lp·z·Ltᵀ). The renewal NGM
# then varies in time through contacts as well as antibody: N(t) uses that week's C*ₜ.
# (Set true for the pooled one-C*-per-window preliminary.)
cfg  = FrameworkConfig(constant_contacts = false)
grid = cis_age_grid()

# Read the CoMix contact data ONCE and reuse it across every window (avoids re-reading/
# re-joining the full Arrow per origin×horizon). Then roll the forecast origin over the
# whole period the current datasets support ("available period"): each origin needs a
# 12-week fit/lag window back to the first inc2prev week, and contact data out to
# origin+3 for the contact-updated iterate. `available_forecast_origins` derives the range.
raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

println("contact data span  : ", extrema(skipmissing(raw.craw.date)))
println("forecast origins   : ", length(wins), " weekly, ",
        first(FORECAST_ORIGINS), " … ", last(FORECAST_ORIGINS))
let w = wins[1], wd0 = load_window_data(wins[1]; grid = grid)
    println("origin[1] fit weeks: ", w.fit_weeks[1], " … ", w.fit_weeks[end])
    println("weekly infections @ origin[1] (age): ", round.(wd0.I_mean[:, end]; digits = 0))
end

In [ ]:
# Fit config: the four combos and the parallel-fit concurrency (CPU- and memory-balanced).
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]
MAX_FIT_CONCURRENCY = fit_concurrency()          # min(threads, cores−1, RAM-budget)
if Threads.nthreads() == 1
    @warn "Julia has 1 thread — pre-fit runs sequentially. Start with JULIA_NUM_THREADS>1 " *
          "(e.g. $(max(1, Sys.CPU_THREADS - 1))) for parallel fitting."
end
println("combos = ", length(combos), " | fit concurrency = ", MAX_FIT_CONCURRENCY,
        " | total fits = ", length(wins) * length(combos) * length(cfg.horizons),
        " (cached ones are skipped)")

In [ ]:
# Detailed implementation lives in 9j_viz_utils.jl — the notebook chooses inputs and calls
# the builders (assembly cache, scoring report, figures). It pulls in 8j_viz_utils.jl too.
include("9j_viz_utils.jl")

# Fixed model order + colour palette shared by the diagnostic figures below (the "four ways").
labels4    = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]
model_cols = [:steelblue, :darkorange, :seagreen, :purple]

# Reload the cached 8j two-stage artefacts and assemble the pooled forecast products (NO re-fit) —
# this origin×combo loop (`two_stage_forecast` → `fit_or_load_stage2` reloads the Stage-2 pooled
# 100×100 draws) is the slow step, so its outputs (`qall` quantile table, `fc_store` fans, `crps`
# cross-check) are CACHED to `../dt_intermediate/9j_assembly_<contacts>.jld2` and reused. Set
# `REBUILD_ASSEMBLY = true` (or delete the cache) to force a fresh reload; the cache self-invalidates
# if origins/combos change. A missing artefact still triggers a fallback re-fit, so run 8j first.
REBUILD_ASSEMBLY = false
asm = assemble_or_load_forecasts(wins, combos, cfg; grid = grid, raw = raw,
                                 save_dir = "../dt_intermediate",
                                 rebuild = REBUILD_ASSEMBLY)
qall, fc_store, crps = asm.qall, asm.fc_store, asm.crps
size(qall)

In [ ]:
scores = score_wis(qall)   # scores both natural & log scale; aggregated by horizon (inst/1e)

# Headline: log-scale WIS by model and by model×horizon + mean native-CRPS cross-check;
# writes the four res/8j_scores_*.csv frames and returns the by-model×horizon log frame.
by_mh_log = report_forecast_scores(scores, wins, crps)

In [ ]:
# log-scale WIS by horizon (one line per model), the four-ways grouped bar (WIS as bar
# height — not the model index, the old bug), and WIS over the forecast period.
save_show(plot_wis_by_horizon(scores, wins, cfg), "../res/8j_wis_by_horizon.png")
save_show(plot_wis_four_ways(scores, labels4, cfg), "../res/8j_wis_four_ways.png")
save_show(plot_wis_over_time(scores), "../res/8j_wis_over_time.png");

In [ ]:
# Relative WIS as a time series (one line per config), faceted by horizon (2×2) — each model's
# log-scale WIS ratioed to negbin|mean per horizon×date (reference on the 1.0 line, <1 = better).
save_show(plot_wis_by_horizon_over_time(scores, labels4, model_cols, cfg),
          "../res/8j_wis_logscale_by_horizon_over_time.png");

In [ ]:
# Forecast vs observed at 9 evenly-spaced origins: one observed series (fit-week history ++
# realized targets) overlaid with the four configs' total-infection fans (median + 90% band).
save_show(plot_forecast_panels(fc_store, wins, labels4, model_cols, cfg; grid = grid, n = 9),
          "../res/8j_forecast_vs_observed_panels.png");

In [ ]:
# Fitted transmission structure over the origins: susceptibility & infectivity as RATIOS to the
# 2-15 group (16-49, >50) from the Stage-2 pooled draws, the separable-GP length-scales
# ρ_diag / ρ_gap (age-yrs) and ρ_time (weeks) from the Stage-1 chains, and the per-contact
# secondary attack rate γ_SAR (one line per model; comparable across origins under un-normalised C*).
tr = collect_transmission_structure(labels4, FORECAST_ORIGINS; grid = grid, h = 1)
save_show(plot_ratio(tr.susc, labels4, FORECAST_ORIGINS, "8j — susceptibility ratio to 2-15 (h=1)"),
          "../res/8j_susc_ratio_over_time.png")
save_show(plot_ratio(tr.inf, labels4, FORECAST_ORIGINS, "8j — infectivity ratio to 2-15 (h=1)"),
          "../res/8j_infectivity_ratio_over_time.png")
save_show(plot_lengthscales(tr.rho, labels4, FORECAST_ORIGINS; h = 1),
          "../res/8j_lengthscale_rho_over_time.png")
save_show(plot_gamma(tr.gamma, labels4, model_cols, FORECAST_ORIGINS; h = 1),
          "../res/8j_gamma_over_time.png");

## Paper-style forecast diagnostics (Munday et al. 2023)

Reproduces the evaluation figures of `inst/pcbi.1011453.pdf` for our four-ways forecast
grid. Reference model = `unweighted-negbin|mean` (the "no-interaction" analog); relative
WIS and coverage use the **log scale** (the headline; robust to the neighbourhood-NGM
natural-scale blow-up).

- **Periods** (Table 2) — named UK COVID phases for aggregating skill over the timeline.
- **Fig 3** — relative WIS, bias, and age-stratified relative WIS by horizon.
- **Fig 4** — relative WIS vs horizon, faceted per pandemic period.
- **Fig 5** — 50% / 90% central-interval coverage (calibration).
- **Reproduction number** — dominant eigenvalue of the origin-week NGM over time, per
  model, with a 90% credible band (`res/9j_reproduction_number.png`).

Outputs are written to `res/9j_*.png` and shown inline.

In [ ]:
# Named pandemic periods (Munday 2023, Table 2) used to aggregate skill over the timeline.
# (9j_viz_utils.jl — the period map + paper-style builders — is already included above.)
period_summary(FORECAST_ORIGINS);

In [ ]:
# Fig 3 analog — (A) relative WIS & (B) bias by horizon, and (C) age-stratified relative
# WIS by horizon (one panel per CIS bin). rWIS < 1 ⇒ better than the negbin-mean reference.
save_show(plot_rwis_bias_by_horizon(scores, labels4, model_cols, cfg),
          "../res/9j_rwis_bias_by_horizon.png")
save_show(plot_rwis_by_age_horizon(scores, labels4, model_cols, grid, cfg),
          "../res/9j_rwis_by_age_horizon.png");

In [ ]:
# Fig 4 analog — relative WIS vs horizon, faceted by pandemic period (origins mapped via
# period_of; log-scale WIS averaged over the origins in each period, then ratioed to the ref).
save_show(plot_rwis_by_period(scores, labels4, model_cols, cfg),
          "../res/9j_rwis_by_horizon_per_period.png");

In [ ]:
# Fig 5 analog — empirical 50% / 90% central-interval coverage by horizon (calibration).
save_show(plot_interval_coverage(scores, labels4, model_cols, cfg),
          "../res/9j_coverage_50_90.png");

In [ ]:
# Reproduction number over time — dominant eigenvalue of the frozen origin-week NGM at the
# 1-week-ahead fit (h=1), per pooled draw. One panel, one STEP per model (R held constant
# across its week) with a 90% band, over the inc2prev national R (England) reference and the
# R = 1 threshold. Reloads the Stage-2 pooled files read-only (no re-fit); this per-origin×combo
# loop is slow, so `rt_store` is CACHED to `../dt_intermediate/9j_rt_<contacts>_h<h>.jld2` and
# reused. Set `REBUILD_RT = true` (or delete the cache) to force a fresh compute; it self-invalidates
# if origins/combos change.
REBUILD_RT = false
rt_store = reproduction_over_time_or_load(combos, labels4, wins, cfg; grid = grid, raw = raw,
                                          h = 1, rebuild = REBUILD_RT)
save_show(plot_reproduction(rt_store, labels4, model_cols, FORECAST_ORIGINS; h = 1),
          "../res/9j_reproduction_number.png");

## §4 Notes

- **Available period**: the forecast origin is rolled weekly over the full span the current
  datasets support — bounded below by the first inc2prev week (the 12-week fit window) and
  above by the CoMix contact-data end (the iterate needs contacts to origin+4). Scores are
  written per model, per model×horizon, and per model×origin (`res/8j_scores_by_model*.csv`).
- **Four ways** = the 2×2 grid; the headline metric is **log-scale WIS aggregated by horizon**
  across all origins (`res/8j_scores_by_model_horizon.csv`, `scale=="log"`), with
  over/under-prediction & dispersion components, bias, and 50/90% coverage.
- **Two-stage cut inference** (inst/4_cut_Bayes.md): 8j fits **Stage 1** (contact-degree GP,
  `model_degree`) once per (degree × origin × horizon), then **Stage 2** (infection block,
  `model_transmission`) conditioning on each of 100 Stage-1 draws (100 Stage-2 draws each). This
  notebook reloads the pooled 10 000-draw predictive per origin×combo×horizon and scores it; a
  missing artefact triggers a fallback re-fit, so run 8j first. The raw CoMix tables are read once.
- **Per-week contact degree, temporally smoothed** (`cfg.constant_contacts = false`, the default
  here): the age-pair mean is estimated for each of the 12 window weeks by a **separable
  spatio-temporal GP** — the age-pair RBF (`ρ_diag`/`ρ_gap`) coupled across weeks by a temporal
  RBF (`ρ_time`), all shared, giving a matrix-normal field `η·Lp·z·Ltᵀ`; the overall level is a
  scalar intercept plus a decoupled temporal-level GP `cₜ = c + σ_c·(Lt·z_c)`. Each age-pair is
  thus a temporally-correlated GP (one shared `ρ_time`), **not** an independent weekly draw;
  dispersion stays per week × block (not smoothed). The renewal NGM is time-varying through
  contacts too: `N(t)` uses that week's `C*ₜ`, and the forecast uses the origin-week slice
  `C*[end]`. Set `true` for the pooled one-`C*`-per-window preliminary.
- **Reciprocity + GP smoothing of the mean** (inst/1e): the contact mean is a *symmetric*
  log-rate over the 28 unordered age pairs, `log μ_{i→j} = r_{min,max} + log(popⱼ)` (so
  `popᵢ·μ_{i→j} = popⱼ·μ_{j→i}` exactly), `r` a **separable spatio-temporal GP** over age
  midpoints (70+ → 74.5) × weeks (shared `ρ_diag`,`ρ_gap`,`ρ_time`; non-centred). The
  neighbourhood NGM stays reciprocity-balanced (size-biased C0 is not reciprocal even when μ is).
- **Secondary attack rate γ_SAR + un-normalised C*** (inst/4_cut_Bayes.md): the `-gnorm` C*/S̄
  normalisation was reverted, so C* feeds the NGM at its raw level and `gamma_sar` is the
  per-contact SAR (`N_ab = γ_SAR · susc_a(1+(F-1)A_a) · C*_ab · inf_b`); it is comparable across
  origins.
- **Group-contact duration weight** (inst/1e): `:cnt_mass=="mass"` contacts (no recorded
  duration) get `cfg.w_dur_group = 2.5/240` (fixed now, estimable later); counts unchanged.
- **Neighbourhood-degree NGM among non-zero** (inst/1c, 1d): `C0 = ⟨k²⟩/⟨k⟩ × g`,
  `g = 1/(1−P₀)` (NegBin, floored) / `g = (1−p⁰)` (hurdle-Weibull); empty per-week Weibull cells
  (⟨k⟩=0) return `C0=0`. Mean NGM (`C0 = ⟨k⟩`) unchanged.
- **Contact-updated iterate** (inst/1d): infections/antibody frozen at each origin; the contact
  matrix is re-estimated each horizon (window ending origin+h); renewal stepped one week at a
  time (mean-plugged lags).
- **WIS on a log scale** (inst/1e): `transform_forecasts(fun=log_shift, offset=1)`; both scales
  written, log is the headline. Stage 1 uses **Pathfinder** (or NUTS via 8j's `STAGE1_USE_NUTS`);
  Stage 2 uses Pathfinder.
- Remaining lean simplifications: **per-week dispersion** is not temporally smoothed (a natural
  next extension), the temporal kernel is a stationary RBF, and the transmission block and 5-day
  generation interval are reference values. Revisit before scientific interpretation.